# Trích keyframe video HCMAI24 trên Colab

**Cách làm:** ffmpeg giải mã video (dùng GPU NVDEC nếu nhanh hơn) và lấy mẫu `SAMPLE_FPS` frame/giây
→ CLIP ViT-B-32 trên GPU tính embedding từng frame mẫu → giữ frame làm **keyframe** khi nó khác
keyframe trước đủ nhiều (`cos < SIM_THRESH`), hoặc khi đã quá `MAX_GAP` giây chưa có keyframe nào
(cảnh tĩnh, ví dụ người dẫn bản tin).

Khác script `get_keyframes.py` của baseline: baseline chạy **ViT-H-14-378 trên mọi frame thứ 6** chỉ để
lọc trùng (vài triệu lượt chạy model lớn). Ở đây chỉ dùng model nhỏ ở 2 fps. Model lớn để dành cho
bước embedding sau, chỉ chạy trên keyframe đã chọn.

**Kết quả trong `OUT_DIR` (Google Drive, giữ lại qua các session):**

| Đường dẫn | Nội dung |
|---|---|
| `keyframes/L01_V001.tar` | ảnh keyframe `L01_V001/<frame_idx:06d>.jpg` (gói tar cho mỗi video: ghi hàng trăm nghìn file nhỏ lên Drive rất chậm) |
| `map-keyframes/L01_V001.csv` | `n, pts_time, fps, frame_idx`: cùng định dạng với BTC, `frame_idx` là số frame thật trong video (dùng khi nộp DRES) |
| `features-vit-b-32/L01_V001.npy` | embedding CLIP B-32 của các keyframe (float16, L2-normalized, cùng thứ tự với map), đủ để chạy baseline ngay |
| `progress.csv` | thống kê từng video đã xong |

**Checkpoint:** chạy lại cell 1–5 rồi cell 8, notebook sẽ làm tiếp từ video chưa xong. Video đã xong
(có đủ tar, npy và csv trên Drive) được bỏ qua. Video lỗi được ghi vào `errors.csv` và tự thử lại. Muốn làm lại
video cụ thể thì dùng cell 8a. Chi tiết ở mục 8.

**Trình tự:** `Runtime → Change runtime type → GPU (T4)` → cell 1 (cài đặt) → 2 (cấu hình) → 3 (Drive)
→ 4 (danh sách video) → 5 (hàm xử lý) → 6 (đo tốc độ giải mã) → 7 (**tinh chỉnh ngưỡng trên 1 video**)
→ 8 (chạy toàn bộ) → 9 (thống kê).

## 1. Cài đặt

In [ ]:
!nvidia-smi -L
!pip install -q open_clip_torch
!ffmpeg -hide_banner -hwaccels

## 2. Cấu hình

In [ ]:
# ===================== CAU HINH -- chi sua o cell nay =====================
MODE = 'drive_zip'      # 'drive_zip' | 'remote_zip' | 'folder'

# MODE='drive_zip': file zip nam trong Drive cua ban. Neu zip la link share cua nguoi khac:
#   mo link -> "Add shortcut to Drive" (shortcut khong ton dung luong Drive) -> dien duong dan.
ZIP_PATHS = [
    '/content/drive/MyDrive/batch1.zip',
    '/content/drive/MyDrive/batch2.zip',
]
# MODE='remote_zip': doc thang zip qua link share (Anyone with the link), tai tung video bang HTTP Range.
ZIP_LINKS = [
    'https://drive.google.com/file/d/PASTE_ID_BATCH1/view?usp=sharing',
    'https://drive.google.com/file/d/PASTE_ID_BATCH2/view?usp=sharing',
]
# MODE='folder': thu muc chua .mp4 (tim de quy), vd video da giai nen san trong Drive / tren may.
VIDEO_DIR = '/content/drive/MyDrive/HCMAI24/videos'

OUT_DIR = '/content/drive/MyDrive/HCMAI24_keyframes'   # ket qua -- nen de trong Drive de khong mat
WORK_DIR = '/content/work'                              # nhap tren SSD local cua Colab

ONLY = None            # vd ['L01', 'L13_V005'] -> chi xu ly video co ten bat dau bang cac tien to nay
PREFETCH = 2           # so video tai/giai nen truoc trong luc dang xu ly video hien tai
SKIP_FAILED = False    # True -> bo qua video da loi o lan chay truoc (ghi trong errors.csv) thay vi thu lai

# --- Lay mau & chon keyframe ---
SAMPLE_FPS = 2.0       # so frame/giay dua vao CLIP de so sanh
SIM_THRESH = 0.85      # cos(frame, keyframe truoc) < nguong -> keyframe moi. Tang -> nhieu keyframe hon
MIN_GAP = 0.5          # giay: 2 keyframe khong gan nhau hon khoang nay
MAX_GAP = 5.0          # giay: canh tinh van lay it nhat 1 keyframe moi MAX_GAP giay
BLACK_THRESH = 8       # bo frame gan nhu den (do sang TB < nguong, thang 0-255); None = tat

# --- Anh luu ---
SAVE_MAX_H = 480       # chieu cao toi da cua anh luu (px). 480 du cho CLIP; tang len 720 neu can OCR tu anh
JPEG_QUALITY = 88

# --- Model so sanh frame (nho, nhanh); embedding cua keyframe cung duoc luu lai ---
CLIP_MODEL = 'ViT-B-32'
CLIP_PRETRAINED = 'laion2b_s34b_b79k'
BATCH = 128            # so frame / lan dua vao GPU

HWACCEL = 'auto'       # 'auto' (do toc do o cell 6 roi chon) | 'cuda' | None (giai ma bang CPU)
BENCH_SEC = 60         # so giay video dung de do toc do o cell 6

## 3. Mount Drive và hàm đọc zip

In [ ]:
import os, sys, io, re, html, time, shutil, zipfile
import requests

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and any(str(p).startswith('/content/drive')
                    for p in [OUT_DIR, VIDEO_DIR, *ZIP_PATHS] if p):
    from google.colab import drive
    drive.mount('/content/drive')


def extract_file_id(link: str) -> str:
    """Lay file-id tu moi dang link Google Drive (hoac tra lai chinh no neu da la id)."""
    link = link.strip()
    for pat in (r'/file/d/([A-Za-z0-9_-]{10,})', r'[?&]id=([A-Za-z0-9_-]{10,})', r'/d/([A-Za-z0-9_-]{10,})'):
        m = re.search(pat, link)
        if m:
            return m.group(1)
    if re.fullmatch(r'[A-Za-z0-9_-]{10,}', link):
        return link
    raise ValueError(f'Khong tim thay file-id trong: {link!r}')


def resolve_gdrive(file_id: str):
    """Tra ve (session, direct_url) da vuot qua trang canh bao virus-scan cua Drive."""
    s = requests.Session()
    r = s.get('https://drive.usercontent.google.com/download',
              params={'id': file_id, 'export': 'download', 'confirm': 't'},
              stream=True, allow_redirects=True)
    if 'text/html' in r.headers.get('Content-Type', ''):
        page = r.text
        r.close()
        form = re.search(r'<form[^>]+action="([^"]+)"(.*?)</form>', page, re.S)
        if not form:
            raise RuntimeError('Drive khong tra ve file: chua share public hoac vuot quota tai '
                               '-> dung MODE="drive_zip" voi shortcut.')
        fields = dict(re.findall(r'name="([^"]+)"\s+value="([^"]*)"', form.group(2)))
        r = s.get(html.unescape(form.group(1)), params=fields, stream=True, allow_redirects=True)
    url = r.url
    r.close()
    return s, url


class HTTPFile(io.RawIOBase):
    """File-object chi doc, seek duoc, lay du lieu bang HTTP Range request."""

    def __init__(self, url, session=None):
        self.url, self.sess, self._pos = url, session or requests.Session(), 0
        r = self.sess.get(url, headers={'Range': 'bytes=0-0'}, stream=True)
        cr = r.headers.get('Content-Range')
        r.close()
        if not cr:
            raise RuntimeError('Server khong ho tro HTTP Range.')
        self.size = int(cr.rsplit('/', 1)[-1])

    def seekable(self): return True
    def readable(self): return True
    def tell(self): return self._pos

    def seek(self, offset, whence=io.SEEK_SET):
        base = {io.SEEK_SET: 0, io.SEEK_CUR: self._pos, io.SEEK_END: self.size}[whence]
        self._pos = max(0, min(base + offset, self.size))
        return self._pos

    def readinto(self, buf):
        n = min(len(buf), self.size - self._pos)
        if n <= 0:
            return 0
        r = self.sess.get(self.url, headers={'Range': f'bytes={self._pos}-{self._pos + n - 1}'}, timeout=120)
        r.raise_for_status()
        data = r.content
        buf[:len(data)] = data
        self._pos += len(data)
        return len(data)


_REMOTE = {}   # zip index -> (session, url); xoa de resolve lai khi link het han


def open_zip(i: int) -> zipfile.ZipFile:
    if MODE == 'drive_zip':
        return zipfile.ZipFile(ZIP_PATHS[i])
    if i not in _REMOTE:
        _REMOTE[i] = resolve_gdrive(extract_file_id(ZIP_LINKS[i]))
    sess, url = _REMOTE[i]
    return zipfile.ZipFile(io.BufferedReader(HTTPFile(url, sess), buffer_size=8 << 20))


print('IN_COLAB =', IN_COLAB, '| MODE =', MODE)

## 4. Danh sách video

In [ ]:
import csv
from collections import Counter
from pathlib import Path


def _wanted(vid):
    return not ONLY or any(vid.startswith(p) for p in ONLY)


VIDEOS = []    # moi phan tu: {'vid', 'path'} (folder) hoac {'vid', 'zip', 'member', 'size'} (zip)
if MODE == 'folder':
    for p in sorted(Path(VIDEO_DIR).rglob('*.mp4')):
        if not p.name.startswith('._') and _wanted(p.stem):
            VIDEOS.append({'vid': p.stem, 'path': str(p), 'size': p.stat().st_size})
else:
    n_zip = len(ZIP_PATHS if MODE == 'drive_zip' else ZIP_LINKS)
    for i in range(n_zip):
        with open_zip(i) as z:
            for info in z.infolist():
                name = info.filename
                stem = os.path.splitext(os.path.basename(name))[0]
                if (name.lower().endswith('.mp4') and not info.is_dir() and '__MACOSX' not in name
                        and not os.path.basename(name).startswith('._') and _wanted(stem)):
                    VIDEOS.append({'vid': stem, 'zip': i, 'member': name, 'size': info.file_size})
VIDEOS.sort(key=lambda it: it['vid'])

dup = [v for v, c in Counter(it['vid'] for it in VIDEOS).items() if c > 1]
assert not dup, f'Trung ten video giua cac nguon: {sorted(dup)[:10]}'


# --- Checkpoint: trang thai luu ngay trong OUT_DIR, khong phu thuoc session ---
FEAT_DIR = 'features-' + CLIP_MODEL.lower().replace('/', '-')
ERRORS_CSV = os.path.join(OUT_DIR, 'errors.csv')


def out_paths(vid):
    return {'tar': os.path.join(OUT_DIR, 'keyframes', f'{vid}.tar'),
            'npy': os.path.join(OUT_DIR, FEAT_DIR, f'{vid}.npy'),
            'csv': os.path.join(OUT_DIR, 'map-keyframes', f'{vid}.csv')}


def is_done(vid):
    """Video xong khi du ca 3 file (csv ghi sau cung nen thieu csv = dang do)."""
    return all(os.path.exists(p) for p in out_paths(vid).values())


def failed_videos():
    """{vid: loi gan nhat} cua cac video tung loi va hien van chua xong."""
    out = {}
    if os.path.exists(ERRORS_CSV):
        with open(ERRORS_CSV, newline='', encoding='utf-8') as f:
            for r in csv.DictReader(f):
                out[r['vid']] = r['error']
    return {v: m for v, m in out.items() if not is_done(v)}


def reset_video(vid):
    """Xoa ket qua cua 1 video de lam lai (xoa csv truoc: video lap tuc bi coi la chua xong)."""
    for key in ('csv', 'npy', 'tar'):
        p = out_paths(vid)[key]
        if os.path.exists(p):
            os.remove(p)


def checkpoint_status():
    done = [it['vid'] for it in VIDEOS if is_done(it['vid'])]
    failed = failed_videos()
    todo = [it['vid'] for it in VIDEOS if not is_done(it['vid'])]
    print(f'{len(VIDEOS)} video, {sum(it["size"] for it in VIDEOS) / 1e9:.1f} GB | da xong {len(done)} | '
          f'con {len(todo)} | tung loi {len(failed)}')
    if done:
        print('  video xong gan nhat (theo thu tu):', done[-1])
    if todo:
        print('  se lam tiep tu      :', todo[:5], '...' if len(todo) > 5 else '')
    for v, m in list(failed.items())[:10]:
        print(f'  loi lan truoc: {v}: {m[:150]}')


checkpoint_status()

## 5. Hàm xử lý

- `iter_frames`: ffmpeg chỉ xuất frame có số thứ tự `n` chia hết cho `step`. Vì vậy
  `frame_idx = k * step` đúng với số frame thật trong video, không bị lệch như khi lấy mẫu theo thời gian.
- `encode_frames`: resize, crop và chuẩn hoá ảnh **trên GPU** (Colab chỉ có 2 vCPU, xử lý ảnh bằng PIL sẽ là
  điểm nghẽn).
- `KeyframeSelector`: so frame hiện tại với **keyframe gần nhất đã giữ** (không phải frame liền trước), nên
  bắt được cả cảnh lia máy chậm.

In [ ]:
import csv, math, tarfile, tempfile, subprocess
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import cv2
import torch
import torch.nn.functional as F
import open_clip

try:
    FFMPEG = shutil.which('ffmpeg') or __import__('imageio_ffmpeg').get_ffmpeg_exe()
except ImportError:
    raise RuntimeError('Khong tim thay ffmpeg (Colab co san; tren may: cai ffmpeg hoac pip install imageio-ffmpeg)')


def _ffmpeg_version():
    out = subprocess.run([FFMPEG, '-version'], capture_output=True, text=True).stdout
    m = re.search(r'version\s+n?(\d+)\.(\d+)', out)
    return (int(m.group(1)), int(m.group(2))) if m else (99, 0)   # ban build git -> coi nhu moi


# Giu nguyen timestamp sau select (khong de ffmpeg chen frame trung cho du fps)
_SYNC = ['-fps_mode', 'passthrough'] if _ffmpeg_version() >= (5, 1) else ['-vsync', '0']

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model, _, _ = open_clip.create_model_and_transforms(CLIP_MODEL, pretrained=CLIP_PRETRAINED, device=DEVICE)
model.eval()
if DEVICE == 'cuda':
    model = model.half()
_pp = getattr(model.visual, 'preprocess_cfg', None) or {}
_sz = model.visual.image_size
IMG_SIZE = _sz[0] if isinstance(_sz, (tuple, list)) else _sz
MEAN = torch.tensor(_pp.get('mean', open_clip.constants.OPENAI_DATASET_MEAN), device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor(_pp.get('std', open_clip.constants.OPENAI_DATASET_STD), device=DEVICE).view(1, 3, 1, 1)
print(f'ffmpeg {".".join(map(str, _ffmpeg_version()))} | {CLIP_MODEL} tren {DEVICE} | input {IMG_SIZE}px')


def video_meta(path):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    if not fps or fps <= 0 or w <= 0 or h <= 0:
        raise RuntimeError(f'Khong doc duoc thong tin video: {path}')
    return fps, n, w, h


def out_size(w, h):
    if SAVE_MAX_H and h > SAVE_MAX_H:
        w, h = w * SAVE_MAX_H / h, SAVE_MAX_H
    return int(round(w / 2)) * 2, int(h) // 2 * 2


def iter_frames(path, step, W, H, hwaccel=None, duration=None):
    """Sinh cac frame RGB uint8 [H, W, 3] co so thu tu chia het cho step."""
    cmd = [FFMPEG, '-nostdin', '-hide_banner', '-loglevel', 'error']
    if hwaccel:
        cmd += ['-hwaccel', hwaccel]
    if duration:
        cmd += ['-t', str(duration)]
    cmd += ['-i', path, '-map', '0:v:0',
            '-vf', f'select=not(mod(n\\,{step})),scale={W}:{H}:flags=area',
            *_SYNC, '-f', 'rawvideo', '-pix_fmt', 'rgb24', 'pipe:1']
    nbytes = W * H * 3
    with tempfile.TemporaryFile() as err:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=err, bufsize=nbytes * 2)
        finished = False
        try:
            while True:
                buf = proc.stdout.read(nbytes)
                if len(buf) < nbytes:
                    break
                yield np.frombuffer(buf, np.uint8).reshape(H, W, 3)
            finished = True
        finally:
            if not finished:
                proc.kill()
            proc.stdout.close()
            rc = proc.wait()
        if rc != 0:
            err.seek(0)
            raise RuntimeError(f'ffmpeg loi (rc={rc}) voi {os.path.basename(path)}:\n'
                               + err.read().decode(errors='replace')[-1500:])


@torch.no_grad()
def encode_frames(frames):
    """frames uint8 [B, H, W, 3] -> (embedding L2-norm [B, D] float32, do sang TB [B] thang 0-255)."""
    x = torch.from_numpy(frames).to(DEVICE).permute(0, 3, 1, 2).float().div_(255)
    bright = x.mean(dim=(1, 2, 3)) * 255
    h, w = x.shape[-2:]
    s = IMG_SIZE / min(h, w)
    x = F.interpolate(x, size=(max(IMG_SIZE, round(h * s)), max(IMG_SIZE, round(w * s))),
                      mode='bicubic', align_corners=False, antialias=True)
    top, left = (x.shape[-2] - IMG_SIZE) // 2, (x.shape[-1] - IMG_SIZE) // 2
    x = (x[..., top:top + IMG_SIZE, left:left + IMG_SIZE] - MEAN) / STD
    e = model.encode_image(x.half() if DEVICE == 'cuda' else x)
    return F.normalize(e.float(), dim=-1), bright


class KeyframeSelector:
    """Quyet dinh tung frame mau co thanh keyframe hay khong (xu ly tuan tu theo thoi gian)."""

    def __init__(self, thresh=None, min_gap=None, max_gap=None, black=None):
        self.thresh = SIM_THRESH if thresh is None else thresh
        self.min_gap = MIN_GAP if min_gap is None else min_gap
        self.max_gap = MAX_GAP if max_gap is None else max_gap
        self.black = BLACK_THRESH if black is None else black
        self.last_e, self.last_t = None, -1e9

    def __call__(self, t, e, bright):
        if self.black is not None and bright < self.black:
            return False, float('nan')
        sim = float('nan') if self.last_e is None else float(e @ self.last_e)
        gap = t - self.last_t
        keep = self.last_e is None or gap >= self.max_gap or (gap >= self.min_gap and sim < self.thresh)
        if keep:
            self.last_e, self.last_t = e, t
        return keep, sim


def _save_jpg(path, rgb):
    ok, buf = cv2.imencode('.jpg', cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
    with open(path, 'wb') as f:
        f.write(buf.tobytes())


def process_video(vid, path, hwaccel=None, trace=False):
    """Trich keyframe cua 1 video vao WORK_DIR/kf/<vid>. Tra ve dict ket qua (chua ghi len OUT_DIR)."""
    fps, n_frames, w, h = video_meta(path)
    step = max(1, round(fps / SAMPLE_FPS))
    W, H = out_size(w, h)
    kf_dir = os.path.join(WORK_DIR, 'kf', vid)
    shutil.rmtree(kf_dir, ignore_errors=True)
    os.makedirs(kf_dir)

    sel = KeyframeSelector()
    rows, embs, futs, batch = [], [], [], []
    tr = {'t': [], 'sim': [], 'kept': [], 'emb': [], 'bright': []} if trace else None
    n_read = 0

    def flush():
        e, b = encode_frames(np.stack(batch))
        e, b = e.cpu().numpy(), b.cpu().numpy()
        for j, frame in enumerate(batch):
            fi = (n_read - len(batch) + j) * step
            t = fi / fps
            keep, sim = sel(t, e[j], b[j])
            if trace:
                for key, val in zip(tr, (t, sim, keep, e[j].astype(np.float16), b[j])):
                    tr[key].append(val)
            if keep:
                rows.append((len(rows) + 1, round(t, 3), fps, fi))
                embs.append(e[j])
                futs.append(writer.submit(_save_jpg, os.path.join(kf_dir, f'{fi:06d}.jpg'), frame))
        batch.clear()

    with ThreadPoolExecutor(2) as writer:
        for frame in iter_frames(path, step, W, H, hwaccel):
            batch.append(frame)
            n_read += 1
            if len(batch) == BATCH:
                flush()
        if batch:
            flush()
        for f in futs:
            f.result()

    expected = math.ceil(n_frames / step) if n_frames > 0 else n_read
    stats = {'vid': vid, 'fps': round(fps, 3), 'n_frames': n_frames, 'step': step, 'W': W, 'H': H,
             'n_sampled': n_read, 'n_kf': len(rows), 'duration_s': round(max(n_frames, n_read * step) / fps, 1)}
    if n_read < 0.95 * expected:
        print(f'   !! {vid}: chi doc duoc {n_read}/{expected} frame mau -- video co the bi hong o cuoi')
    if trace:
        tr = {k: np.array(v) for k, v in tr.items()}
    return {'kf_dir': kf_dir, 'rows': rows,
            'embs': np.array(embs, dtype=np.float16).reshape(len(embs), -1),
            'stats': stats, 'trace': tr}


def _copy(src, dst):
    """Copy roi doi ten -> file o dich luon hoan chinh (khong de lai file do dang khi bi ngat)."""
    shutil.copyfile(src, dst + '.part')
    os.replace(dst + '.part', dst)


def publish(vid, res):
    """Ghi ket qua 1 video len OUT_DIR. map-keyframes/<vid>.csv ghi sau cung = danh dau da xong."""
    for sub in ('keyframes', 'map-keyframes', FEAT_DIR):
        os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)
    tmp = os.path.join(WORK_DIR, 'publish')
    os.makedirs(tmp, exist_ok=True)

    tar_path = os.path.join(tmp, f'{vid}.tar')
    with tarfile.open(tar_path, 'w') as tf:
        tf.add(res['kf_dir'], arcname=vid)
    _copy(tar_path, os.path.join(OUT_DIR, 'keyframes', f'{vid}.tar'))
    mb = os.path.getsize(tar_path) / 1e6

    npy_path = os.path.join(tmp, f'{vid}.npy')
    np.save(npy_path, res['embs'])
    _copy(npy_path, os.path.join(OUT_DIR, FEAT_DIR, f'{vid}.npy'))

    csv_path = os.path.join(tmp, f'{vid}.csv')
    with open(csv_path, 'w', newline='') as f:
        wr = csv.writer(f)
        wr.writerow(['n', 'pts_time', 'fps', 'frame_idx'])
        wr.writerows(res['rows'])
    _copy(csv_path, os.path.join(OUT_DIR, 'map-keyframes', f'{vid}.csv'))

    for p in (tar_path, npy_path, csv_path):
        os.remove(p)
    shutil.rmtree(res['kf_dir'], ignore_errors=True)
    return mb


def tmp_video_path(vid):
    return os.path.join(WORK_DIR, 'videos', f'{vid}.mp4')


def fetch_video(item):
    """Tra ve duong dan .mp4 tren may; voi zip thi giai nen rieng video do ra WORK_DIR."""
    if 'path' in item:
        return item['path']
    dst = tmp_video_path(item['vid'])
    if os.path.exists(dst) and os.path.getsize(dst) == item['size']:
        return dst
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    for attempt in range(4):
        try:
            with open_zip(item['zip']) as z, z.open(item['member']) as src, open(dst + '.part', 'wb') as out:
                shutil.copyfileobj(src, out, 16 << 20)
            os.replace(dst + '.part', dst)
            return dst
        except Exception as e:
            if attempt == 3:
                raise
            print(f'   (tai {item["vid"]} loi: {type(e).__name__}: {e} -- thu lai)')
            _REMOTE.pop(item['zip'], None)     # link remote co the het han -> resolve lai
            time.sleep(5 * (attempt + 1))


os.makedirs(WORK_DIR, exist_ok=True)
print('San sang.')

## 6. Đo tốc độ giải mã: CPU hay GPU (NVDEC)

Giải mã video là phần tốn thời gian nhất, vì phải giải mã **mọi** frame dù chỉ giữ 2 frame/giây.
Cell này giải mã `BENCH_SEC` giây đầu của video đầu tiên theo từng cách rồi chọn cách nhanh hơn.

In [ ]:
item0 = VIDEOS[0]
path0 = fetch_video(item0)
fps0, n0, w0, h0 = video_meta(path0)
step0 = max(1, round(fps0 / SAMPLE_FPS))
print(f'{item0["vid"]}: {w0}x{h0} @ {fps0:.2f} fps, {n0} frame ({n0 / fps0 / 60:.1f} phut) '
      f'-> lay 1/{step0} frame, luu {out_size(w0, h0)}')

bench = {}
for hw in ([None, 'cuda'] if HWACCEL == 'auto' else [HWACCEL]):
    t0 = time.time()
    try:
        n = sum(1 for _ in iter_frames(path0, step0, *out_size(w0, h0), hwaccel=hw, duration=BENCH_SEC))
    except Exception as e:
        print(f'  {str(hw):5s}: khong dung duoc ({str(e).strip().splitlines()[-1][:120]})')
        continue
    dt = time.time() - t0
    bench[hw] = dt
    print(f'  {str(hw):5s}: {BENCH_SEC}s video trong {dt:.1f}s = {BENCH_SEC / dt:.1f}x realtime '
          f'({n} frame mau, ~{n * step0 / dt:.0f} fps giai ma)')

BEST_HW = min(bench, key=bench.get) if bench else None
print('>>> Dung hwaccel =', BEST_HW)

## 7. Tinh chỉnh ngưỡng trên 1 video (nên làm trước khi chạy toàn bộ)

Biểu đồ cho thấy độ giống giữa từng frame mẫu và keyframe gần nhất, cùng các điểm được chọn làm keyframe.
Bảng bên dưới **mô phỏng lại** số keyframe với các `SIM_THRESH` khác nhau mà không phải giải mã lại video.
Mục tiêu là mỗi cảnh có ít nhất 1 keyframe, đồng thời không giữ quá nhiều frame gần như giống hệt nhau. Với tin tức HCMAI,
thường rơi vào khoảng 20–40 keyframe/phút (bản tin cắt cảnh nhanh sẽ ở mức cao).

In [ ]:
import matplotlib.pyplot as plt

t0 = time.time()
res0 = process_video(item0['vid'], path0, BEST_HW, trace=True)
st0, tr0 = res0['stats'], res0['trace']
dt = time.time() - t0
print(f'{item0["vid"]}: {st0["n_kf"]} keyframe / {st0["n_sampled"]} frame mau | '
      f'{st0["n_kf"] / (st0["duration_s"] / 60):.1f} kf/phut | {dt:.0f}s ({st0["duration_s"] / dt:.1f}x realtime)')

print('\nSIM_THRESH  so_kf  kf/phut')
for th in (0.80, 0.85, 0.88, 0.90, 0.92, 0.94, 0.96):
    s = KeyframeSelector(thresh=th)
    n = sum(s(t, e.astype(np.float32), b)[0] for t, e, b in zip(tr0['t'], tr0['emb'], tr0['bright']))
    print(f'   {th:.2f}    {n:5d}   {n / (st0["duration_s"] / 60):6.1f}' + ('   <- hien tai' if th == SIM_THRESH else ''))

fig, ax = plt.subplots(figsize=(16, 3.5))
ax.plot(tr0['t'], tr0['sim'], lw=0.7, label='cos voi keyframe gan nhat')
kept = tr0['kept'].astype(bool)
ax.scatter(tr0['t'][kept], np.ones(kept.sum()), s=6, c='r', label='keyframe')
ax.axhline(SIM_THRESH, ls='--', c='gray', lw=0.8)
ax.set_xlabel('giay'); ax.set_ylim(0.5, 1.02); ax.legend(loc='lower right')
plt.show()

files = sorted(os.listdir(res0['kf_dir']))[:24]
fig, axes = plt.subplots(3, 8, figsize=(20, 7))
for ax in axes.flat:
    ax.axis('off')
for ax, fn in zip(axes.flat, files):
    ax.imshow(cv2.cvtColor(cv2.imread(os.path.join(res0['kf_dir'], fn)), cv2.COLOR_BGR2RGB))
    ax.set_title(f'frame {int(fn[:-4])} ({int(fn[:-4]) / fps0:.1f}s)', fontsize=8)
plt.tight_layout(); plt.show()
kb = np.mean([os.path.getsize(os.path.join(res0['kf_dir'], f)) for f in os.listdir(res0['kf_dir'])]) / 1e3
est_gb = kb * 1e3 * st0['n_kf'] / item0['size'] * sum(it['size'] for it in VIDEOS) / 1e9
print(f'Anh trung binh {kb:.0f} KB -> uoc tinh toan bo ~{est_gb:.1f} GB (ngoai suy theo dung luong video)')
# Ket qua video nay chua ghi len Drive; cell 8 se xu ly lai voi cau hinh cuoi cung.

## 8. Chạy toàn bộ (có checkpoint)

**Checkpoint nằm ngay trong `OUT_DIR` trên Drive**, không phụ thuộc session:

- Video được coi là xong khi có đủ 3 file `keyframes/<vid>.tar`, `features-*/<vid>.npy` và
  `map-keyframes/<vid>.csv`. File csv được ghi sau cùng, nên nếu bị ngắt giữa chừng thì video đó
  chỉ đang dở và sẽ được làm lại từ đầu, mất khoảng 1 phút.
- Bị ngắt (mất kết nối, hết giờ, bấm Stop) thì chạy lại các cell 1–5 rồi tới cell này. Có thể bỏ qua
  cell 6 và 7, khi đó giải mã bằng CPU, trừ khi đặt `HWACCEL='cuda'`. Cell 4 in ra video sẽ làm tiếp.
- Video bị lỗi được ghi vào `errors.csv` và **tự thử lại** ở lần chạy sau. Đặt `SKIP_FAILED = True`
  để bỏ qua chúng, ví dụ khi video bị hỏng thật.
- Muốn **làm lại** một số video đã xong (ví dụ sau khi đổi ngưỡng): dùng cell 8a ngay bên dưới.

In [ ]:
# 8a. LAM LAI video da xong: dien ten/tien to roi chay cell nay, sau do chay cell 8.
# Cell nay chi xoa ket qua cu; nho de lai [] de lan chay lai sau khong xoa nham.
REDO_NOW = []          # vd ['L01_V003', 'L02']  (theo tien to, giong ONLY)

for it in VIDEOS:
    if REDO_NOW and any(it['vid'].startswith(p) for p in REDO_NOW) and is_done(it['vid']):
        reset_video(it['vid'])
        print('da xoa ket qua cu ->', it['vid'])
checkpoint_status()

In [ ]:
HW = (globals().get('BEST_HW') if HWACCEL == 'auto' else HWACCEL)
skip = set(failed_videos()) if SKIP_FAILED else set()
todo = [it for it in VIDEOS if not is_done(it['vid']) and it['vid'] not in skip]
print(f'Con {len(todo)}/{len(VIDEOS)} video (bo qua {len(skip)} video loi) | hwaccel={HW} | '
      f'SIM_THRESH={SIM_THRESH} SAMPLE_FPS={SAMPLE_FPS}')
if todo:
    print('Bat dau tu:', todo[0]['vid'])

PROGRESS = os.path.join(OUT_DIR, 'progress.csv')
os.makedirs(OUT_DIR, exist_ok=True)


def _append_csv(path, row):
    new = not os.path.exists(path)
    with open(path, 'a', newline='', encoding='utf-8') as f:
        wr = csv.DictWriter(f, fieldnames=list(row))
        if new:
            wr.writeheader()
        wr.writerow(row)


fetcher = ThreadPoolExecutor(1)
pending, errors = {}, []
t_all, secs = time.time(), []
current = None

try:
    for k, item in enumerate(todo):
        for j in range(k, min(k + 1 + PREFETCH, len(todo))):
            if j not in pending:
                pending[j] = fetcher.submit(fetch_video, todo[j])
        vid = current = item['vid']
        try:
            path = pending.pop(k).result()
            t0 = time.time()
            res = process_video(vid, path, HW)
            mb = publish(vid, res)
            st = res['stats']
            st.update(sec=round(time.time() - t0, 1), tar_mb=round(mb, 1))
            _append_csv(PROGRESS, st)
            secs.append(time.time() - t0)
            eta = (time.time() - t_all) / (k + 1) * (len(todo) - k - 1)
            print(f'[{k + 1}/{len(todo)}] {vid}: {st["n_kf"]:4d} kf / {st["n_sampled"]:5d} mau | '
                  f'{st["duration_s"] / 60:5.1f} phut video, {st["sec"]:5.0f}s '
                  f'({st["duration_s"] / max(st["sec"], 1e-6):4.1f}x) | {mb:5.1f} MB | '
                  f'ETA {eta / 3600:.1f}h')
        except Exception as e:
            msg = f'{type(e).__name__}: {e}'
            errors.append((vid, msg))
            _append_csv(ERRORS_CSV, {'vid': vid, 'time': time.strftime('%Y-%m-%d %H:%M:%S'), 'error': msg[:2000]})
            print(f'!! {vid}: {msg[:300]}')
        finally:
            if 'path' not in item and os.path.exists(tmp_video_path(vid)):
                os.remove(tmp_video_path(vid))
        current = None
finally:
    fetcher.shutdown(wait=False, cancel_futures=True)
    if current:
        print(f'\n>>> Dung giua video {current} (chua xong, se lam lai tu dau). '
              f'Chay lai cell nay de tiep tuc.')
    print(f'\nXong {len(secs)} video trong {(time.time() - t_all) / 3600:.2f}h | loi {len(errors)}')
    for vid, msg in errors:
        print('  ', vid, msg[:200])

## 9. Thống kê kết quả

In [ ]:
import pandas as pd

n_done = sum(is_done(it['vid']) for it in VIDEOS)
print(f'Da xong {n_done}/{len(VIDEOS)} video')
PROGRESS = os.path.join(OUT_DIR, 'progress.csv')
if os.path.exists(PROGRESS):
    df = pd.read_csv(PROGRESS).drop_duplicates('vid', keep='last')
    print(f'Tong keyframe: {df.n_kf.sum():,} | video: {df.duration_s.sum() / 3600:.1f}h | '
          f'TB {df.n_kf.sum() / (df.duration_s.sum() / 60):.1f} kf/phut | '
          f'anh: {df.tar_mb.sum() / 1e3:.2f} GB | toc do TB {df.duration_s.sum() / df.sec.sum():.1f}x realtime')
    df['kf_per_min'] = df.n_kf / (df.duration_s / 60)
    print('\nVideo it keyframe nhat (nen xem lai):')
    print(df.nsmallest(5, 'kf_per_min')[['vid', 'n_kf', 'duration_s', 'kf_per_min']].to_string(index=False))
    print('\nVideo nhieu keyframe nhat:')
    print(df.nlargest(5, 'kf_per_min')[['vid', 'n_kf', 'duration_s', 'kf_per_min']].to_string(index=False))

## 10. (Tuỳ chọn) Giải nén keyframe để dùng ở bước tiếp theo

Bước embedding bằng model lớn (ViT-H, SigLIP…) cần ảnh dạng file. Nên giải nén ra SSD local
(`/content/...` trên Colab, hoặc ổ SSD trên laptop), **không** giải nén vào Drive.

In [ ]:
RUN_UNTAR = False
DEST = '/content/keyframes'

if RUN_UNTAR:
    import glob
    tars = sorted(glob.glob(os.path.join(OUT_DIR, 'keyframes', '*.tar')))
    for i, t in enumerate(tars, 1):
        with tarfile.open(t) as tf:
            tf.extractall(DEST, **({'filter': 'data'} if hasattr(tarfile, 'data_filter') else {}))
        if i % 50 == 0 or i == len(tars):
            print(f'{i}/{len(tars)}')
    print('->', DEST)